# 🏗️ 베이스라인 모델 구축: 5클래스 대화 분류

---

## 프로젝트 개요

| 항목 | 내용 |
|---|---|
| **과제** | 대화의 성격을 5개 클래스 중 하나로 분류 |
| **클래스** | `협박 대화`, `갈취 대화`, `직장 내 괴롭힘 대화`, `기타 괴롭힘 대화`, `일반 대화` |
| **평가 지표** | Macro F1 Score |
| **모델** | KLUE-RoBERTa-base (Fine-tuning) |
| **데이터** | `baseline.csv` — 위협 4클래스(증강) + 일반 대화(합성) |

### 참조 문서
- `docs/DLThon.md`: 대회 규칙 및 평가 항목
- `docs/strategy.md`: EDA 기반 데이터 전략 보고서
- `reports/eda_results.txt`: EDA 분석 수치
- `model_plan.md`: 모델 구현 계획서

---

## 1. 환경 설정 및 라이브러리 임포트

학습에 필요한 모든 라이브러리를 로드합니다.

| 라이브러리 | 용도 |
|---|---|
| `transformers` | KLUE-RoBERTa 모델 및 토크나이저 |
| `torch` | 모델 정의, 학습, 추론 |
| `sklearn` | 데이터 분할, F1 Score 계산, 혼동 행렬 |
| `pandas`, `numpy` | 데이터 처리 |
| `matplotlib`, `seaborn` | 시각화 |

In [ ]:
# ============================================================
# Cell 1: 환경 설정 및 라이브러리 임포트
# ============================================================

# --- 기본 라이브러리 ---
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')

# --- PyTorch ---
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# --- HuggingFace Transformers ---
from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup

# --- 평가 ---
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# --- 시각화 ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- 한글 폰트 설정 (Windows 기준) ---
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# --- 재현성 보장을 위한 시드 고정 ---
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

# --- GPU 확인 ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')
if device.type == 'cuda':
    print(f'GPU 이름: {torch.cuda.get_device_name(0)}')
    print(f'GPU 메모리: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

---

## 2. 데이터 로드 및 전처리

### 2-1. 데이터 로드

`baseline.csv`는 팀원이 다음 작업을 완료한 최종 학습 데이터입니다:
- **위협 4클래스**: 원본 데이터 + 증강(동의어 교체, 역번역 등)
- **일반 대화**: LLM 기반 합성 데이터 (~1,000건)
- **중복 제거**: 완전 중복(104건) + 준-중복(117건) 제거 완료

### 2-2. 텍스트 전처리 전략

| 처리 항목 | 결정 | 근거 (strategy.md) |
|---|---|---|
| `\n` (줄바꿈) | `[SEP]` 토큰으로 치환 | 발화 턴 경계를 모델이 인식하도록 |
| `!`, `?` | 유지 | 클래스 구분 힌트 (협박: !=1.85, 직장: !=0.70) |
| 이모티콘 (`ㅋㅋ`, `ㅠㅠ`) | 별도 처리 불필요 | 학습 데이터 전체에서 0건 |
| 조사 (`을`, `를` 등) | 유지 | BERT 계열 모델이 문맥 정보로 활용 가능 |

In [ ]:
# ============================================================
# Cell 2: 데이터 로드 및 전처리
# ============================================================

# --- 2-1. 데이터 로드 ---
# baseline.csv: 팀원이 증강/합성을 완료한 최종 학습 데이터
# 컬럼 구성: idx, class, conversation
train_df = pd.read_csv('data/baseline.csv')

print(f'전체 데이터 크기: {train_df.shape}')
print(f'컬럼: {train_df.columns.tolist()}')
print(f'\n=== 클래스 분포 ===')
print(train_df['class'].value_counts())
print(f'\n총 {len(train_df)}건')

In [ ]:
# ============================================================
# Cell 3: 레이블 인코딩 + 텍스트 전처리
# ============================================================

# --- 레이블 매핑 ---
# 5개 클래스를 정수로 인코딩합니다.
# 순서는 원본 데이터의 클래스 등장 순서를 기준으로 합니다.
label2id = {
    '갈취 대화': 0,
    '기타 괴롭힘 대화': 1,
    '직장 내 괴롭힘 대화': 2,
    '협박 대화': 3,
    '일반 대화': 4,
}
id2label = {v: k for k, v in label2id.items()}

# 레이블 컬럼 생성
train_df['label'] = train_df['class'].map(label2id)

# 매핑 검증: NaN이 있으면 매핑 실패한 클래스가 있다는 의미
assert train_df['label'].isna().sum() == 0, \
    f"매핑 실패 클래스 발견: {train_df[train_df['label'].isna()]['class'].unique()}"
print('레이블 매핑 완료 (NaN 없음)')
print(f'클래스 수: {len(label2id)}개')
for name, idx in label2id.items():
    print(f'  {idx}: {name}')

In [ ]:
# ============================================================
# Cell 4: 텍스트 전처리 함수
# ============================================================

def preprocess(text):
    """
    대화 텍스트를 모델 입력에 적합한 형태로 변환합니다.
    
    전처리 전략 (strategy.md 기반):
    1. 줄바꿈(\n)을 [SEP] 토큰으로 치환
       → RoBERTa가 대화의 발화 턴 경계를 인식할 수 있도록 합니다.
       → EDA [1-2]: 대화당 평균 10턴이므로, [SEP]은 약 10개 삽입됩니다.
    2. 연속 공백을 단일 공백으로 정리
    3. 앞뒤 공백 제거
    
    의도적으로 제거하지 않는 것:
    - 느낌표(!): 협박(1.85) > 갈취(1.02) → 클래스 구분 힌트
    - 물음표(?): 전 클래스 4.08~4.87로 고르게 분포
    - 조사(을/를): BERT 계열 모델이 문맥 정보로 활용
    """
    # Step 1: 줄바꿈 → [SEP] 토큰 치환 (발화 턴 경계 표시)
    text = re.sub(r'\n+', ' [SEP] ', text)
    
    # Step 2: 연속 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# 전처리 적용
train_df['text'] = train_df['conversation'].apply(preprocess)

# 전처리 결과 확인
print('=== 전처리 전 (원본) ===')
print(train_df['conversation'].iloc[0][:200])
print(f'\n=== 전처리 후 ===')
print(train_df['text'].iloc[0][:200])
print(f'\n전처리 완료: {len(train_df)}건')

---

## 3. Train / Validation 분할

학습 안정성과 일반화 성능을 평가하기 위해 데이터를 **80:20** 비율로 분할합니다.

| 파라미터 | 설정 | 이유 |
|---|---|---|
| 분할 비율 | 80:20 | 데이터가 ~5,000건으로 적으므로 검증 셋이 최소 1,000건은 확보되어야 함 |
| 층화 추출 (Stratify) | `label` 기준 | 클래스 비율이 Train/Val에서 동일하게 유지되도록 |
| Random Seed | 42 | 재현성 보장 |

In [ ]:
# ============================================================
# Cell 5: Train/Validation 분할
# ============================================================

# Stratified Split: 각 클래스의 비율이 Train/Val에서 동일하게 유지
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'].tolist(),     # 전처리된 텍스트
    train_df['label'].tolist(),    # 정수 레이블
    test_size=0.2,                 # 검증 셋 20%
    random_state=SEED,             # 재현성
    stratify=train_df['label'],    # 층화 추출
)

print(f'Train 크기: {len(train_texts)}건')
print(f'Val   크기: {len(val_texts)}건')

# 분할 후 클래스 비율 검증
print(f'\n=== Train 클래스 분포 ===')
train_dist = pd.Series(train_labels).value_counts().sort_index()
for idx, count in train_dist.items():
    print(f'  {id2label[idx]}: {count}건 ({count/len(train_labels)*100:.1f}%)')

print(f'\n=== Val 클래스 분포 ===')
val_dist = pd.Series(val_labels).value_counts().sort_index()
for idx, count in val_dist.items():
    print(f'  {id2label[idx]}: {count}건 ({count/len(val_labels)*100:.1f}%)')

---

## 4. 토크나이저 및 Dataset 정의

### 4-1. 토크나이저: KLUE-RoBERTa

KLUE-RoBERTa의 전용 토크나이저(BPE 기반)를 사용합니다.  
한국어 텍스트를 서브워드 단위로 분절하여 OOV(미등록 단어) 문제를 최소화합니다.

### 4-2. MAX_LEN = 256 설정 근거

| EDA 수치 | 값 |
|---|---|
| 글자 수 중간값 | 203자 |
| 글자 수 75% | 270자 |
| 글자 수 최대 | 874자 |

- RoBERTa 토크나이저는 한국어 1글자당 약 1~2 토큰을 생성
- **256 토큰**이면 대부분의 대화(75%)를 완전히 커버
- 512 대비 **메모리 50% 절약** → 배치 사이즈 증가 가능

In [ ]:
# ============================================================
# Cell 6: 토크나이저 및 하이퍼파라미터 설정
# ============================================================

# --- 모델 이름 ---
# KLUE-RoBERTa-base: 한국어 특화 사전학습 모델
# - BERT 대비 Dynamic Masking + NSP 제거로 문맥 파악 능력 향상
# - 110M 파라미터로 Ablation Study 반복에 적합한 크기
MODEL_NAME = 'klue/roberta-base'

# --- 하이퍼파라미터 ---
MAX_LEN = 256       # 토큰 최대 길이 (EDA: 글자 중간값 203, 75%=270)
BATCH_SIZE = 32     # 배치 크기 (MAX_LEN=256 기준 GPU 메모리 최적)
LEARNING_RATE = 2e-5  # Fine-tuning 표준 범위 (1e-5 ~ 5e-5)
EPOCHS = 5          # 학습 에폭 (데이터 ~5,000건 → 3~5 에폭에서 수렴 예상)
WARMUP_RATIO = 0.1  # 학습 초기 10%는 워밍업 (불안정 방지)
WEIGHT_DECAY = 0.01 # L2 정규화 계수 (과적합 억제)
DROPOUT_RATE = 0.3  # 분류 헤드 드롭아웃 (데이터 적으므로 0.1보다 높게)

# --- 토크나이저 로드 ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f'모델: {MODEL_NAME}')
print(f'MAX_LEN: {MAX_LEN}')
print(f'BATCH_SIZE: {BATCH_SIZE}')
print(f'LEARNING_RATE: {LEARNING_RATE}')
print(f'EPOCHS: {EPOCHS}')
print(f'DROPOUT_RATE: {DROPOUT_RATE}')
print(f'Vocab Size: {tokenizer.vocab_size}')

# --- 토크나이저 동작 확인 ---
sample_text = train_texts[0][:100]
tokens = tokenizer.tokenize(sample_text)
print(f'\n=== 토크나이저 동작 확인 ===')
print(f'원문 ({len(sample_text)}자): {sample_text}')
print(f'토큰 ({len(tokens)}개): {tokens[:20]}...')

In [ ]:
# ============================================================
# Cell 7: PyTorch Dataset 클래스 정의
# ============================================================

class ConversationDataset(Dataset):
    """
    대화 텍스트를 KLUE-RoBERTa 입력 형식으로 변환하는 Dataset 클래스.
    
    각 샘플은 다음으로 구성됩니다:
    - input_ids: 토큰 ID 시퀀스 (0~vocab_size)
    - attention_mask: 실제 토큰(1) vs 패딩(0) 구분
    - label: 정수 레이블 (0~4)
    """
    
    def __init__(self, texts, labels, tokenizer, max_len):
        """
        Args:
            texts (list[str]): 전처리된 텍스트 리스트
            labels (list[int]): 정수 레이블 리스트
            tokenizer: HuggingFace 토크나이저
            max_len (int): 최대 토큰 수
        """
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        # 텍스트를 토큰화하여 모델 입력 형식으로 변환
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,      # 최대 길이 제한
            padding='max_length',         # 짧은 시퀀스는 패딩
            truncation=True,              # 긴 시퀀스는 잘라냄
            return_tensors='pt',          # PyTorch 텐서 반환
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),         # (MAX_LEN,)
            'attention_mask': encoding['attention_mask'].squeeze(0), # (MAX_LEN,)
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
        }


# --- Dataset 생성 ---
train_dataset = ConversationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = ConversationDataset(val_texts, val_labels, tokenizer, MAX_LEN)

# --- DataLoader 생성 ---
# Train: shuffle=True (매 에폭마다 데이터 순서 섞기)
# Val: shuffle=False (평가 시에는 순서 유지)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train Dataset: {len(train_dataset)}건 → {len(train_loader)} 배치')
print(f'Val   Dataset: {len(val_dataset)}건 → {len(val_loader)} 배치')

# --- 배치 형태 확인 ---
sample_batch = next(iter(train_loader))
print(f'\n=== 배치 형태 확인 ===')
print(f"input_ids shape:      {sample_batch['input_ids'].shape}")       # (BATCH, MAX_LEN)
print(f"attention_mask shape: {sample_batch['attention_mask'].shape}")  # (BATCH, MAX_LEN)
print(f"label shape:          {sample_batch['label'].shape}")           # (BATCH,)

---

## 5. 모델 정의

### 아키텍처 구조

```
[입력 텍스트]
    │
    ▼
KLUE-RoBERTa-base (768 hidden dim)
    │
    ▼
[CLS] 토큰 출력 (768차원) ← 대화 전체의 의미를 응축한 벡터
    │
    ▼
Dropout (p=0.3) ← 과적합 방지 (데이터 ~5,000건으로 적음)
    │
    ▼
Linear (768 → 5) ← 5클래스 분류
    │
    ▼
[Logits] → CrossEntropyLoss (내부적으로 Softmax 포함)
```

### 설계 결정 근거

| 결정 | 선택 | 근거 |
|---|---|---|
| 풀링 방식 | `[CLS]` 토큰 | 문장 분류 태스크 표준. Mean Pooling 대비 안정적 |
| Dropout | 0.3 | 데이터가 ~5,000건으로 적어 과적합 위험 높음 → 기본(0.1)보다 높게 |
| 분류 헤드 | 단일 Linear | 베이스라인에서는 단순 구조 → 성능 분석 후 MLP 헤드 추가 검토 |

In [ ]:
# ============================================================
# Cell 8: 분류 모델 정의
# ============================================================

class ConversationClassifier(nn.Module):
    """
    KLUE-RoBERTa 기반 대화 분류 모델.
    
    구조:
    1. KLUE-RoBERTa backbone → 문맥을 이해한 토큰별 벡터 출력
    2. [CLS] 토큰 추출 → 대화 전체의 의미를 768차원 벡터로 응축
    3. Dropout → 학습 시 랜덤하게 뉴런을 비활성화하여 과적합 방지
    4. Linear → 768차원 → 5클래스 로짓값 출력
    """
    
    def __init__(self, model_name, num_classes=5, dropout_rate=0.3):
        super().__init__()
        
        # 사전학습된 KLUE-RoBERTa 로드 (가중치 포함)
        self.backbone = AutoModel.from_pretrained(model_name)
        
        # 분류 헤드: Dropout → Linear
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(
            self.backbone.config.hidden_size,  # 768 (base 모델)
            num_classes                         # 5 (분류 클래스 수)
        )
    
    def forward(self, input_ids, attention_mask):
        """
        Args:
            input_ids: (batch_size, max_len) - 토큰 ID
            attention_mask: (batch_size, max_len) - 패딩 마스크
        
        Returns:
            logits: (batch_size, num_classes) - 각 클래스의 로짓값
        """
        # Step 1: RoBERTa에 입력 → 토큰별 hidden state 출력
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        
        # Step 2: [CLS] 토큰(첫 번째 토큰)의 hidden state 추출
        # last_hidden_state shape: (batch_size, max_len, hidden_size)
        # [:, 0, :] → (batch_size, hidden_size) = (batch_size, 768)
        cls_output = outputs.last_hidden_state[:, 0, :]
        
        # Step 3: Dropout 적용 (학습 시에만 활성화, 추론 시 비활성화)
        cls_output = self.dropout(cls_output)
        
        # Step 4: Linear 분류 → 5클래스 로짓값
        logits = self.classifier(cls_output)  # (batch_size, 5)
        
        return logits


# --- 모델 인스턴스 생성 ---
model = ConversationClassifier(
    model_name=MODEL_NAME,
    num_classes=len(label2id),
    dropout_rate=DROPOUT_RATE,
).to(device)

# --- 모델 파라미터 수 확인 ---
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'전체 파라미터: {total_params:,}개')
print(f'학습 가능 파라미터: {trainable_params:,}개')
print(f'모델 구조:\n{model}')

---

## 6. 학습 설정

### 옵티마이저 & 스케줄러

| 구성 요소 | 선택 | 근거 |
|---|---|---|
| **옵티마이저** | AdamW | BERT fine-tuning 표준. Weight Decay를 적용하여 L2 정규화 |
| **스케줄러** | Linear Warmup + Decay | 초기 10% 구간은 LR을 서서히 올려 학습 안정화 |
| **손실 함수** | CrossEntropyLoss | 다중 클래스 분류 표준 |
| **Gradient Clipping** | `max_norm=1.0` | 그래디언트 폭발 방지 |

In [ ]:
# ============================================================
# Cell 9: 옵티마이저, 스케줄러, 손실 함수 설정
# ============================================================

# --- 옵티마이저: AdamW ---
# AdamW = Adam + Weight Decay (L2 정규화)
# BERT/RoBERTa fine-tuning의 사실상 표준 옵티마이저
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,        # 2e-5
    weight_decay=WEIGHT_DECAY,  # 0.01
)

# --- 스케줄러: Linear Warmup + Decay ---
# 학습 초기 10%는 LR을 0에서 서서히 올림 (워밍업) → 이후 선형 감소
# 이유: 사전학습된 가중치를 갑자기 큰 LR로 업데이트하면 붕괴 위험
total_steps = len(train_loader) * EPOCHS  # 전체 학습 스텝 수
warmup_steps = int(total_steps * WARMUP_RATIO)  # 워밍업 스텝 수

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# --- 손실 함수: CrossEntropyLoss ---
# 다중 클래스 분류의 표준 손실 함수
# 내부적으로 Softmax + NLLLoss를 결합
#
# [참고] 클래스 불균형이 심할 경우 아래처럼 가중치 적용 가능:
# class_counts = train_df['label'].value_counts().sort_index().values
# class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
# class_weights = class_weights / class_weights.sum() * len(class_weights)
# criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
criterion = nn.CrossEntropyLoss()

print(f'Total steps: {total_steps}')
print(f'Warmup steps: {warmup_steps}')
print(f'Optimizer: AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})')
print(f'Scheduler: Linear Warmup + Decay')
print(f'Loss: CrossEntropyLoss')

---

## 7. 학습 실행

### 학습 루프 구조

각 에폭마다 다음을 수행합니다:

1. **학습(Train)**: 전체 Train 데이터를 순회하며 모델 가중치 업데이트
2. **검증(Validation)**: 전체 Val 데이터에 대해 손실과 F1 Score 측정
3. **Best Model 저장**: Val F1이 갱신되면 모델 가중치를 저장

### 핵심 기법

| 기법 | 설명 |
|---|---|
| **Gradient Clipping** | 그래디언트 크기를 `max_norm=1.0`으로 제한하여 폭발 방지 |
| **Best Model 저장** | Val F1 기준으로 최고 성능 모델만 보존 (Early Stopping 대용) |

In [ ]:
# ============================================================
# Cell 10: 학습 & 검증 함수 정의
# ============================================================

def train_epoch(model, loader, criterion, optimizer, scheduler, device):
    """
    1 에폭 학습을 수행합니다.
    
    Returns:
        avg_loss (float): 평균 손실
        f1 (float): Macro F1 Score
    """
    model.train()  # 학습 모드 (Dropout 활성화)
    total_loss = 0
    all_preds, all_trues = [], []
    
    for batch in loader:
        # --- 데이터를 GPU로 전송 ---
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        # --- Forward pass ---
        optimizer.zero_grad()  # 이전 그래디언트 초기화
        logits = model(input_ids, attention_mask)  # (batch, 5)
        loss = criterion(logits, labels)
        
        # --- Backward pass ---
        loss.backward()  # 그래디언트 계산
        
        # Gradient Clipping: 그래디언트가 1.0을 초과하면 스케일링
        # → 학습 안정성 확보 (RoBERTa fine-tuning 시 권장)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()   # 가중치 업데이트
        scheduler.step()   # 러닝 레이트 스케줄 업데이트
        
        # --- 메트릭 수집 ---
        total_loss += loss.item()
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())  # 예측 클래스
        all_trues.extend(labels.cpu().numpy())                 # 정답 클래스
    
    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_trues, all_preds, average='macro')  # Macro F1
    return avg_loss, f1


def evaluate(model, loader, criterion, device):
    """
    검증/테스트를 수행합니다.
    
    Returns:
        avg_loss (float): 평균 손실
        f1 (float): Macro F1 Score
        all_preds (list): 전체 예측값
        all_trues (list): 전체 정답값
    """
    model.eval()  # 평가 모드 (Dropout 비활성화)
    total_loss = 0
    all_preds, all_trues = [], []
    
    with torch.no_grad():  # 그래디언트 계산 비활성화 (메모리/속도 절약)
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
            all_trues.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_trues, all_preds, average='macro')
    return avg_loss, f1, all_preds, all_trues


print('학습/검증 함수 정의 완료')

In [ ]:
# ============================================================
# Cell 11: 메인 학습 루프 실행
# ============================================================

best_f1 = 0         # 최고 Val F1 기록
history = []         # 에폭별 학습 기록

print('=' * 70)
print(f'학습 시작 | 모델: {MODEL_NAME} | 디바이스: {device}')
print(f'Train: {len(train_dataset)}건 | Val: {len(val_dataset)}건 | Epochs: {EPOCHS}')
print('=' * 70)

for epoch in range(EPOCHS):
    # --- 학습 ---
    train_loss, train_f1 = train_epoch(
        model, train_loader, criterion, optimizer, scheduler, device
    )
    
    # --- 검증 ---
    val_loss, val_f1, val_preds, val_trues = evaluate(
        model, val_loader, criterion, device
    )
    
    # --- 에폭별 기록 저장 ---
    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_f1': train_f1,
        'val_loss': val_loss,
        'val_f1': val_f1,
        'lr': scheduler.get_last_lr()[0],  # 현재 러닝 레이트
    })
    
    # --- 에폭 결과 출력 ---
    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    print(f'  Train | Loss: {train_loss:.4f} | F1: {train_f1:.4f}')
    print(f'  Val   | Loss: {val_loss:.4f} | F1: {val_f1:.4f}')
    print(f'  LR: {scheduler.get_last_lr()[0]:.2e}')
    
    # --- Best Model 저장 ---
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'  ✅ Best model saved! (Val F1: {best_f1:.4f})')

print('\n' + '=' * 70)
print(f'학습 완료 | Best Val F1: {best_f1:.4f}')
print('=' * 70)

---

## 8. 평가 및 분석

최적 모델(Best Val F1)을 로드하여 상세한 성능 분석을 수행합니다.

### 분석 항목

| 분석 | 목적 |
|---|---|
| **Classification Report** | 클래스별 Precision, Recall, F1 확인 |
| **혼동 행렬** | 어떤 클래스 쌍이 가장 많이 혼동되는지 시각화 |
| **학습 곡선** | 과적합 여부 (Train-Val Loss 격차) 확인 |

### 혼동 행렬 분석 포인트 (strategy.md 기반)
- `협박 ↔ 기타 괴롭힘`: 코사인 유사도 **0.87**로 가장 높음 → 혼동 예상
- `협박 ↔ 갈취`: 코사인 유사도 **0.83** → 두 번째 혼동 위험
- `일반 대화`: 위협 클래스로 오분류되는 비율 → 합성 데이터 품질 지표

In [ ]:
# ============================================================
# Cell 12: Best Model 로드 및 상세 평가
# ============================================================

# --- Best Model 로드 ---
model.load_state_dict(torch.load('best_model.pt'))
print('Best model 로드 완료')

# --- 검증 셋 최종 평가 ---
val_loss, val_f1, final_preds, final_trues = evaluate(
    model, val_loader, criterion, device
)

# --- Classification Report ---
# 클래스별 Precision, Recall, F1을 상세히 확인합니다.
class_names = [id2label[i] for i in range(len(id2label))]
print('\n' + '=' * 60)
print('Classification Report (Validation Set)')
print('=' * 60)
print(classification_report(
    final_trues, final_preds,
    target_names=class_names,
    digits=4,  # 소수점 4자리까지 표시
))
print(f'Overall Macro F1: {val_f1:.4f}')

In [ ]:
# ============================================================
# Cell 13: 혼동 행렬 시각화
# ============================================================

# 혼동 행렬 계산
cm = confusion_matrix(final_trues, final_preds)

# 시각화
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d',
    xticklabels=class_names,
    yticklabels=class_names,
    cmap='Blues',
    cbar_kws={'label': '건수'},
)
plt.title('Confusion Matrix (Validation Set)', fontsize=14)
plt.ylabel('실제 (Actual)', fontsize=12)
plt.xlabel('예측 (Predicted)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# --- 혼동 분석 ---
# 대각선 제외, 오분류가 가장 많은 쌍을 식별
print('\n=== 오분류 분석 (상위 5건) ===')
confusion_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i][j] > 0:
            confusion_pairs.append((
                class_names[i],  # 실제
                class_names[j],  # 예측
                cm[i][j],        # 건수
            ))
confusion_pairs.sort(key=lambda x: -x[2])
for rank, (actual, pred, count) in enumerate(confusion_pairs[:5], 1):
    print(f'  {rank}위: [{actual}] → [{pred}]으로 {count}건 오분류')

In [ ]:
# ============================================================
# Cell 14: 학습 곡선 시각화
# ============================================================

hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Loss 곡선 ---
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], 'o-', label='Train Loss', color='#1f77b4')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'], 'o-', label='Val Loss', color='#ff7f0e')
axes[0].set_title('Loss 곡선', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- F1 Score 곡선 ---
axes[1].plot(hist_df['epoch'], hist_df['train_f1'], 'o-', label='Train F1', color='#2ca02c')
axes[1].plot(hist_df['epoch'], hist_df['val_f1'], 'o-', label='Val F1', color='#d62728')
axes[1].set_title('Macro F1 Score 곡선', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'학습 기록 | Best Val F1: {best_f1:.4f}', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# --- 학습 기록 테이블 출력 ---
print('\n=== 에폭별 학습 기록 ===')
print(hist_df.to_string(index=False, float_format='{:.4f}'.format))

---

## 9. 추론 및 제출 파일 생성

학습 완료된 최적 모델로 `test.csv`를 추론하여 `submission.csv`를 생성합니다.

### 추론 파이프라인

```
test.csv → 전처리(preprocess) → 토큰화 → 모델 추론(eval mode) → argmax → 레이블 디코딩 → submission.csv
```

In [ ]:
# ============================================================
# Cell 15: 추론 및 submission.csv 생성
# ============================================================

# --- 테스트 데이터 로드 ---
test_df = pd.read_csv('data/test.csv')
print(f'테스트 데이터: {len(test_df)}건')
print(f'컬럼: {test_df.columns.tolist()}')

# --- 전처리 적용 (학습 시와 동일한 전처리) ---
test_df['text'] = test_df['conversation'].apply(preprocess)

# --- Dataset & DataLoader 생성 ---
# 테스트에는 레이블이 없으므로 더미 레이블(0)을 사용
test_dataset = ConversationDataset(
    texts=test_df['text'].tolist(),
    labels=[0] * len(test_df),  # 더미 레이블 (사용하지 않음)
    tokenizer=tokenizer,
    max_len=MAX_LEN,
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# --- 최적 모델 로드 ---
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

# --- 추론 실행 ---
all_preds = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        logits = model(input_ids, attention_mask)
        preds = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)

# --- 레이블 디코딩 ---
test_df['class'] = [id2label[p] for p in all_preds]

# --- submission.csv 생성 ---
submission = test_df[['idx', 'class']]
submission.to_csv('submission.csv', index=False)

print('\n=== submission.csv 생성 완료 ===')
print(f'파일 크기: {len(submission)}건')
print(f'\n=== 예측 클래스 분포 ===')
print(submission['class'].value_counts())
print(f'\n=== 상위 5건 미리보기 ===')
print(submission.head())

---

## 10. Ablation Study 실험 로그

### 실험 설계 (model_plan.md §8 기반)

각 실험은 **하나의 변수만 변경**하고 나머지는 베이스라인과 동일하게 유지합니다.

| 실험 | 변수 | 비교 조건 | 기대 효과 |
|---|---|---|---|
| **Exp-A1** | MAX_LEN | 128 / 256 / 512 | 성능 vs 속도 트레이드오프 |
| **Exp-A2** | Dropout | 0.1 / 0.3 / 0.5 | 과적합 정도에 따른 최적값 |
| **Exp-A3** | 전처리 | `\n`→`[SEP]` / `\n`→공백 / 원문 유지 | 발화 구분 방식의 영향 |
| **Exp-A4** | 손실 함수 | CE / Focal Loss / Label Smoothing | 유사 클래스 혼동 개선 |
| **Exp-A5** | 모델 | RoBERTa / KoELECTRA | 모델 아키텍처 비교 |

In [ ]:
# ============================================================
# Cell 16: Ablation Study 결과 기록 템플릿
# ============================================================

# 실험 결과를 기록할 딕셔너리 리스트
ablation_results = [
    {
        'Experiment': 'Baseline',
        'Model': MODEL_NAME,
        'MAX_LEN': MAX_LEN,
        'Dropout': DROPOUT_RATE,
        'Preprocess': '\\n → [SEP]',
        'Loss': 'CrossEntropy',
        'Val_F1': best_f1,
        'Notes': '기준선',
    },
    # --- 아래에 추가 실험 결과를 기록하세요 ---
    # {
    #     'Experiment': 'Exp-A1a',
    #     'Model': MODEL_NAME,
    #     'MAX_LEN': 128,
    #     'Dropout': DROPOUT_RATE,
    #     'Preprocess': '\\n → [SEP]',
    #     'Loss': 'CrossEntropy',
    #     'Val_F1': 0.0000,
    #     'Notes': 'MAX_LEN 축소 실험',
    # },
]

# 결과 테이블 출력
ablation_df = pd.DataFrame(ablation_results)
print('=== Ablation Study 결과 ===')
print(ablation_df.to_string(index=False))

---

## 📋 요약 및 다음 단계

### 베이스라인 결과

| 항목 | 결과 |
|---|---|
| 모델 | KLUE-RoBERTa-base |
| Best Val F1 | (학습 후 기록) |
| 가장 혼동되는 쌍 | (혼동 행렬 분석 후 기록) |

### 다음 단계

1. **혼동 행렬 분석**: 어떤 클래스 쌍이 가장 많이 혼동되는지 파악
2. **Ablation Study**: §10의 실험 설계에 따라 변수별 성능 비교
3. **팀원 피드백 반영**: 합성 데이터 품질에 따라 전처리/학습 전략 조정
4. **최종 제출**: 최고 성능 모델로 `submission.csv` 생성 및 리더보드 업로드